In [1]:
import numpy as np, matplotlib.pyplot as plt
from scipy.sparse.linalg import cg
import pandas as pd

In [2]:
def get_train_labels():
    from torchvision.datasets import MNIST
    from torchvision.transforms import ToTensor

    mnist_train = MNIST("./data/download", train=True, transform=ToTensor())
    return np.array([*mnist_train.targets])

def get_test_labels():
    from torchvision.datasets import MNIST
    from torchvision.transforms import ToTensor

    mnist_test = MNIST("./data/download", train=False, transform=ToTensor())
    return np.array([*mnist_test.targets])

def relabel(arr: np.ndarray, label_0: float, label_1: float) -> np.ndarray:
    return label_0 + arr * (label_1 - label_0)



In [3]:
samples = 4000

Y_tr = relabel(get_train_labels() % 2, -1, +1)
Y_ts = relabel(get_test_labels() % 2, -1, +1)

train_path = f"./data/out/K_train_train_{samples}_samples_60000x60000.npy"
test_path = f"./data/out/K_test_train_{samples}_samples_10000x60000.npy"

In [4]:
def accuracy(N_tr):
    K_tr_inv = np.linalg.inv(np.load(train_path, 'r')[:N_tr, :N_tr])
    K_ts = np.load(test_path, 'r')[:, :N_tr]
    predictions = np.sign(K_ts @ K_tr_inv @ Y_tr[:N_tr])

    return np.mean(predictions == Y_ts)

def accuracy_cg(N_tr, reg: float = 0):
    x, info = cg(
        np.load(train_path, 'r')[:N_tr, :N_tr] + reg * np.eye(N_tr),
        Y_tr[:N_tr]
    )
    K_ts = np.load(test_path, 'r')[:, :N_tr]
    predictions = np.sign(K_ts @ x)

    return np.mean(predictions == Y_ts)

In [5]:
Ns = range(30000, 45001, 5000)
regs = [0.5, 0.7]

df = pd.DataFrame(index= Ns, columns= regs)


train_path = f"./data/out/K_train_train_{samples}_samples_60000x60000.npy"
test_path = f"./data/out/K_test_train_{samples}_samples_10000x60000.npy"
train_path2 = f"./data/out/K_train_train_{samples // 2}_samples_60000x60000.npy"
test_path2 = f"./data/out/K_test_train_{samples // 2}_samples_10000x60000.npy"


K_tr = (np.load(train_path) * samples + np.load(train_path2) * samples/2) / (3/2 * samples)
K_ts = (np.load(test_path) * samples + np.load(test_path2) * samples/2) / (3/2 * samples)

for N_tr in Ns:
    x = Y_tr[:N_tr] * 25
    for reg in regs:
        x, info = cg(
            K_tr[:N_tr, :N_tr] + reg * np.eye(N_tr),
            Y_tr[:N_tr],
            x0= x
        )

        predictions = np.sign(K_ts[:, :N_tr] @ x)

        acc = np.mean(predictions == Y_ts)

        print(f"N = {N_tr}, reg = {reg}, acc = {acc}")

        df.at[N_tr, reg] = np.mean(predictions == Y_ts)

df

N = 30000, reg = 0.5, acc = 0.9829
N = 30000, reg = 0.7, acc = 0.9821
N = 35000, reg = 0.5, acc = 0.9829
N = 35000, reg = 0.7, acc = 0.9827
N = 40000, reg = 0.5, acc = 0.9824


KeyboardInterrupt: 

In [ ]:
df.at[N_tr, reg] = 100
df

In [ ]:
for N_tr in range(100, 2000, 100):
    print(f"N_tr: {N_tr} -> accuracy: {accuracy(N_tr)}")